# Kiểm thử cGAN explainer trên COVID-QU-Ex: COVID-19 vs Normal

Luồng làm việc:
1. Import, cấu hình, **chỉ định checkpoint** (explainer, classifier, tùy chọn Inception cho FID)
2. Lấy code từ GitHub, đăng nhập Hugging Face
3. Tải dữ liệu thành file ảnh + `images.csv` (giống notebook train)
4. Tải checkpoint về và kiểm tra thông tin
5. Tính điểm classifier trên split test bằng `compute_scores.py`
6. Đánh giá định lượng bằng `test.py` (CV, FID)
7. Sinh ảnh phản thực hàng loạt bằng `generate.py`
8. **Đưa một ảnh riêng lẻ vào và nhận dãy ảnh phản thực** 

## 1. Import

In [ ]:
import os, re, sys, json, shutil, hashlib, subprocess
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

import torch

from datasets import load_dataset, Image as HFImage
from huggingface_hub import login, HfApi, list_repo_files, hf_hub_download
from kaggle_secrets import UserSecretsClient
from IPython.display import display, Image as IPImage

## 2. Cấu hình

### 2.1 Chọn checkpoint
- **Explainer**
- **Classifier**
- **Inception cho FID** 

In [ ]:
# Hugging Face
HF_USERNAME = "dacntt-kltn"
DATASET_REPO_ID = f"{HF_USERNAME}/covid-normal-cxr"
RUNS_PREFIX = "training-runs"

# CHECKPOINT EXPLAINER (cGAN)
MODEL_REPO_ID = f"{HF_USERNAME}/cgan-covid-normal-cxr"
EXPLAINER_RUN = None # None: run mới nhất trên Hub hoặc tên thư mục run
EXPLAINER_FILE = "explainer_last.pt" # tên file checkpoint
EXPLAINER_LOCAL_PATH = None # khi file checkpoint đang tồn tại trên môi trường chạy, không cần tải từ hugging face

# CHECKPOINT CLASSIFIER (DenseNet121)
CLASSIFIER_REPO_ID = f"{HF_USERNAME}/densenet121-covid-normal-cxr"
CLASSIFIER_FILENAME = "training-runs/00000-densenet121-covid-normal/best.pt" # None: best.pt mới nhất trên Hub
CLASSIFIER_LOCAL_PATH = None 

# Inception để tính FID 
USE_FINETUNED_FID = True # True: dùng Inception đã fine-tune; False: Inception ImageNet
FID_REPO_ID = f"{HF_USERNAME}/inception-v3-covid-normal-cxr"
FID_RUN = None # None = run mới nhất có file inception_cxr.pt
FID_FILENAME = "inception_cxr.pt"
FID_NET_LOCAL_PATH = None

# Code trên GitHub
GITHUB_REPO_URL = "https://github.com/Vinh306/medical-image-classification-counterfactual-explanation"
CODE_DIR = Path("/kaggle/working/cf-explanation-code")
CODE_SUBDIR = "cGAN"

# Dữ liệu trên đĩa 
DATA_DIR = Path("/tmp/covid-normal-cxr")
IMAGES_DIR = DATA_DIR / "images"
IMAGES_CSV = DATA_DIR / "images.csv" # path, split, label, label_name, covid19
TEST_SPLITS = ["test"]   
OUT_DIR = Path("/kaggle/working/test-outputs")

TARGET = "covid19"
SEED = 42
NUM_WORKERS = 2

# Tham số test.py
MAX_IMAGES = 2100 # số ảnh tối đa mỗi nhóm (âm / dương)
TAU = 0.8 # ngưỡng |f(x) - f(x_c)| để tính là lật nhãn thành công (CV)
TEST_BATCH_SIZE = 32

# Tham số generate.py
GEN_GROUP = "any" # "neg" | "mild" | "pos" | "any"
GEN_N = 8

OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 3. Lấy code từ GitHub

In [ ]:
if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(CODE_DIR)], check=True)

GIT_COMMIT = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"],
                            capture_output=True, text=True, check=True).stdout.strip()
print("Commit đang dùng:", GIT_COMMIT)

CODE_PATH = CODE_DIR / CODE_SUBDIR
sys.path.insert(0, str(CODE_PATH))
import common, data, models, classifier, visualize, metrics
for m in (common, data, models, classifier, visualize, metrics):
    assert str(CODE_DIR) in m.__file__, f"Import nhầm module trùng tên: {m.__name__} -> {m.__file__}"

from common import N_BINS, IMG_SIZE, LABELS, DEVICE, seed_everything
from data import load_image, score_to_bin_np
from classifier import load_target_classifier
from models import load_explainer
from visualize import progressive_counterfactuals, save_strip_png
assert LABELS == [TARGET], f"Repo cần LABELS = ['{TARGET}'] (1 logit), hiện tại: {LABELS}"

seed_everything(SEED)
print("Đã import từ", CODE_PATH, "| IMG_SIZE =", IMG_SIZE, "| N_BINS =", N_BINS, "| device:", DEVICE)

# Chạy script của repo và in log theo thời gian thực
def run_cmd(cmd, cwd=None, check=True):
    cmd = [str(c) for c in cmd]
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    except BaseException:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    if check and rc != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {rc}): {' '.join(cmd)}")
    return rc

## 4. Đăng nhập Hugging Face và chọn checkpoint

In [ ]:
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
api = HfApi()
print("Đăng nhập là:", api.whoami()["name"])

### 4.1 Danh sách checkpoint explainer có trên Hub

In [ ]:
hub_files = list_repo_files(MODEL_REPO_ID, repo_type="model")
run_re = re.compile(rf"^{RUNS_PREFIX}/([0-9]{{5}}-[^/]+)/(explainer_[^/]+\.pt)$")
hub_ckpts = {}
for fn in hub_files:
    m = run_re.match(fn)
    if m:
        hub_ckpts.setdefault(m.group(1), []).append(m.group(2))
for run, fs in sorted(hub_ckpts.items()):
    print(run)
    for f_ in sorted(fs):
        print("   ", f_)
if not hub_ckpts:
    print("Chưa thấy checkpoint nào trên Hub; hãy dùng EXPLAINER_LOCAL_PATH.")

### 4.2 Tải checkpoint về đĩa và kiểm tra thông tin

In [ ]:
# Explainer
if EXPLAINER_LOCAL_PATH:
    EXPLAINER_PATH = Path(EXPLAINER_LOCAL_PATH)
    explainer_src = str(EXPLAINER_PATH)
else:
    run = EXPLAINER_RUN or sorted(hub_ckpts)[-1]
    assert run in hub_ckpts and EXPLAINER_FILE in hub_ckpts[run], \
        f"Không tìm thấy {run}/{EXPLAINER_FILE} trên Hub (xem danh sách ở mục 4.1)"
    explainer_src = f"{MODEL_REPO_ID}/{RUNS_PREFIX}/{run}/{EXPLAINER_FILE}"
    EXPLAINER_PATH = DATA_DIR / "explainer_ckpt.pt"
    shutil.copy(hf_hub_download(MODEL_REPO_ID, f"{RUNS_PREFIX}/{run}/{EXPLAINER_FILE}", repo_type="model"), EXPLAINER_PATH)
assert EXPLAINER_PATH.exists(), f"Không có file {EXPLAINER_PATH}"

# Classifier
if CLASSIFIER_LOCAL_PATH:
    CLASSIFIER_PATH = Path(CLASSIFIER_LOCAL_PATH)
    classifier_src = str(CLASSIFIER_PATH)
else:
    clf_file = CLASSIFIER_FILENAME
    if not clf_file:
        files = list_repo_files(CLASSIFIER_REPO_ID, repo_type="model")
        clf_file = sorted(f for f in files if f.startswith(f"{RUNS_PREFIX}/") and f.endswith("/best.pt"))[-1]
    classifier_src = f"{CLASSIFIER_REPO_ID}/{clf_file}"
    CLASSIFIER_PATH = DATA_DIR / "classifier_best.pt"
    shutil.copy(hf_hub_download(CLASSIFIER_REPO_ID, clf_file, repo_type="model"), CLASSIFIER_PATH)
assert CLASSIFIER_PATH.exists(), f"Không có file {CLASSIFIER_PATH}"

# Inception cho FID
FID_NET_PATH, fid_src = None, "Inception ImageNet"
if FID_NET_LOCAL_PATH:
    FID_NET_PATH = Path(FID_NET_LOCAL_PATH)
    assert FID_NET_PATH.exists(), f"Không có file {FID_NET_PATH}"
    fid_src = str(FID_NET_PATH)
elif USE_FINETUNED_FID:
    fid_files = list_repo_files(FID_REPO_ID, repo_type="model")
    if FID_RUN:
        fid_file = f"{RUNS_PREFIX}/{FID_RUN}/{FID_FILENAME}"
        assert fid_file in fid_files, f"Không thấy {fid_file} trên {FID_REPO_ID}"
    else:
        cands = sorted(f for f in fid_files if f.startswith(f"{RUNS_PREFIX}/") and f.endswith(f"/{FID_FILENAME}"))
        assert cands, f"Chưa có {FID_FILENAME} nào trên {FID_REPO_ID}"
        fid_file = cands[-1]
    FID_NET_PATH = DATA_DIR / "inception_cxr.pt"
    shutil.copy(hf_hub_download(FID_REPO_ID, fid_file, repo_type="model"), FID_NET_PATH)
    fid_src = f"{FID_REPO_ID}/{fid_file}"

if FID_NET_PATH is not None:   # kiểm tra đúng định dạng state_dict thuần mà test.py yêu cầu
    sd = torch.load(FID_NET_PATH, map_location="cpu")
    assert "fc.weight" in sd and "model" not in sd, \
        "FID net phải là state_dict thuần (inception_cxr.pt), không phải best.pt/last.pt"
    del sd

ck = torch.load(EXPLAINER_PATH, map_location="cpu", weights_only=False)
print("Explainer :", explainer_src)
print(f"iter = {ck.get('it')} | keys = {list(ck.keys())}")
tr_args = ck.get("args", {})
if tr_args:
    print("cấu hình train:", {k: tr_args.get(k) for k in
          ["iters", "batch_size", "per_bin", "lr", "n_critic", "lam_gan", "lam_f", "lam_rec"]})
del ck
ckc = torch.load(CLASSIFIER_PATH, map_location="cpu", weights_only=False)
print("Classifier:", classifier_src)
if isinstance(ckc, dict):
    print(f"epoch {ckc.get('epoch')}, {ckc.get('monitor')} = {ckc.get('value')}")
del ckc
print("FID net:", fid_src)

# Tag để cache điểm số theo classifier
CLF_TAG = hashlib.md5(classifier_src.encode()).hexdigest()[:8]
print("CLF_TAG:", CLF_TAG)

## 5. Tải dữ liệu về file cứng và tạo CSV đường dẫn ảnh

In [ ]:
ds = load_dataset(DATASET_REPO_ID, token=token)
print(ds)

label_names = ds["train"].features["label"].names
print("Nhãn:", label_names)
assert label_names == ["normal", "covid19"], "Thứ tự nhãn không đúng (cần normal=0, covid19=1)"
covid_idx = label_names.index("covid19")
for s in TEST_SPLITS:
    assert s in ds, f"Split '{s}' không có trong dataset: {list(ds.keys())}"

def ext_of(b):
    if b[:8] == b"\x89PNG\r\n\x1a\n": return ".png"
    if b[:2] == b"\xff\xd8": return ".jpg"
    return ".png"

rows, n_written = [], 0
for split in TEST_SPLITS:
    d = ds[split].cast_column("image", HFImage(decode=False))
    for i, ex in enumerate(tqdm(d, desc=split)):
        b = ex["image"]["bytes"]
        # Nếu bytes là None, đọc dữ liệu trực tiếp từ đường dẫn file trong cache
        if b is None and ex["image"].get("path"):
            b = Path(ex["image"]["path"]).read_bytes()
            
        assert b is not None, f"{split}[{i}] không có bytes ảnh"
        name = label_names[ex["label"]]
        p = IMAGES_DIR / split / name / f"{i:06d}{ext_of(b)}"
        if not p.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
            tmp = p.with_suffix(p.suffix + ".tmp")
            tmp.write_bytes(b)
            os.replace(tmp, p)
            n_written += 1
        rows.append(dict(path=str(p), split=split, label=int(ex["label"]), label_name=name,
                         covid19=float(ex["label"] == covid_idx)))

images_df = pd.DataFrame(rows)
images_df.to_csv(IMAGES_CSV, index=False)
print(f"\nGhi mới {n_written} file | tổng {len(images_df)} ảnh | CSV: {IMAGES_CSV}")
print(images_df.groupby(["split", "label_name"]).size().unstack(fill_value=0))

# Kiểm tra ảnh đọc được bằng đúng hàm load_image của repo
im = PILImage.open(images_df["path"].iloc[0])
print("Ảnh mẫu:", im.size, im.mode)
if im.size != (IMG_SIZE, IMG_SIZE):
    print(f"CẢNH BÁO: ảnh {im.size} khác {IMG_SIZE}x{IMG_SIZE}; load_image của repo sẽ tự crop giữa và resize.")
for p in images_df["path"].sample(min(8, len(images_df)), random_state=SEED):
    t = load_image(p)
    assert t.shape == (1, IMG_SIZE, IMG_SIZE) and 0 <= t.min() and t.max() <= 1
print("load_image OK:", tuple(t.shape))

## 6. Tính điểm xác suất của classifier (compute_scores.py)

In [ ]:
run_cmd([sys.executable, "compute_scores.py", "--help"], cwd=CODE_PATH)

In [ ]:
SCORES_CSV = DATA_DIR / f"scores_{'+'.join(TEST_SPLITS)}_{CLF_TAG}.csv"

need = not SCORES_CSV.exists() or len(pd.read_csv(SCORES_CSV, usecols=["path"])) != len(images_df)
if need:
    run_cmd([sys.executable, "compute_scores.py",
             "--csv_path", IMAGES_CSV,
             "--classifier_path", CLASSIFIER_PATH,
             "--target", TARGET,
             "--output_path", SCORES_CSV,
             "--batch_size", 64,
             "--num_workers", NUM_WORKERS],
            cwd=CODE_PATH)
else:
    print(f"Đã có {SCORES_CSV}, bỏ qua tính điểm.")

sc = pd.read_csv(SCORES_CSV)
assert {"path", "score", "split", "label"} <= set(sc.columns) and sc["score"].between(0, 1).all()
sc["bin"] = score_to_bin_np(sc["score"].values)

print("AUC của classifier:")
for s in sc["split"].unique():
    g = sc[sc.split == s]
    print(f"{s:>10}: AUC = {roc_auc_score(g['label'], g['score']):.4f} | n = {len(g)}")
print(f"\nNhóm âm (score < 0.2): {(sc.score < 0.2).sum()} ảnh | nhóm dương (score > 0.8): {(sc.score > 0.8).sum()} ảnh")
print("\nSố ảnh mỗi bin:")
print(pd.crosstab(sc["bin"], sc["label_name"]).reindex(range(N_BINS), fill_value=0).to_string())

## 7. Đánh giá định lượng bằng `test.py`

`test.py` lấy nhóm âm (`score < 0.2`) chuyển sang bin cuối (c = 1) và nhóm dương (`score > 0.8`) chuyển sang bin đầu (c = 0), rồi tính:
- **CV** (counterfactual validity): tỉ lệ ảnh có `|f(x) - f(x_c)| > TAU`
- **FID** giữa ảnh gốc và ảnh phản thực

In [ ]:
run_cmd([sys.executable, "test.py", "--help"], cwd=CODE_PATH)

In [ ]:
RESULTS_JSON = OUT_DIR / "results.json"
cmd = [sys.executable, "test.py",
       "--csv_path", SCORES_CSV,
       "--classifier_path", CLASSIFIER_PATH,
       "--explainer_path", EXPLAINER_PATH,
       "--target", TARGET,
       "--max_images", MAX_IMAGES,
       "--tau", TAU,
       "--batch_size", TEST_BATCH_SIZE,
       "--num_workers", NUM_WORKERS,
       "--out_json", RESULTS_JSON,
       "--seed", SEED]
if FID_NET_PATH is not None:
    cmd += ["--fid_net", FID_NET_PATH]
print("Lệnh:", " ".join(str(c) for c in cmd), "\n")
run_cmd(cmd, cwd=CODE_PATH)

In [ ]:
res = json.loads(RESULTS_JSON.read_text())
table = pd.DataFrame({
    "CV (tau=%.2f)" % res["tau"]: [res["CV"], res["CV_neg_to_pos"], res["CV_pos_to_neg"]],
    "FID": [res["FID_total"], res["FID_neg_to_pos"], res["FID_pos_to_neg"]],
}, index=["Tổng", "âm → dương (c=1)", "dương → âm (c=0)"])
print(f"Explainer: {explainer_src}\nClassifier: {classifier_src}\nFID net: {res['fid_net']}")
print(f"n_neg = {res['n_neg']} | n_pos = {res['n_pos']} | n_total = {res['n_total']}\n")
display(table.round(4))

## 8. Sinh ảnh phản thực hàng loạt bằng `generate.py`

In [ ]:
run_cmd([sys.executable, "generate.py", "--help"], cwd=CODE_PATH)

In [ ]:
GEN_DIR = OUT_DIR / f"generate_{GEN_GROUP}"
if GEN_DIR.exists():
    shutil.rmtree(GEN_DIR)
run_cmd([sys.executable, "generate.py",
         "--csv_path", SCORES_CSV,
         "--classifier_path", CLASSIFIER_PATH,
         "--explainer_path", EXPLAINER_PATH,
         "--target", TARGET,
         "--group", GEN_GROUP,
         "--n", GEN_N,
         "--out_dir", GEN_DIR,
         "--seed", SEED],
        cwd=CODE_PATH)

for p in sorted(GEN_DIR.glob("*.png")):
    print(p.name)
    display(IPImage(filename=str(p), width=1800))

display(pd.read_csv(GEN_DIR / "predictions.csv").round(3))

## 9. Đưa một ảnh riêng lẻ vào và nhận dãy ảnh phản thực

### 9.1 Nạp classifier và explainer từ checkpoint đã chọn

In [ ]:
f = load_target_classifier(CLASSIFIER_PATH, TARGET)
X = load_explainer(EXPLAINER_PATH)
print("Đã nạp classifier và explainer trên", DEVICE)

SINGLE_DIR = OUT_DIR / "single"
SINGLE_DIR.mkdir(parents=True, exist_ok=True)

@torch.no_grad()
def explain_image(path, out_png=None, show=True):
    path = str(path)
    x01 = load_image(path)[None].to(DEVICE)                 # (1,1,256,256) trong [0,1]
    xcs, preds, fx = progressive_counterfactuals(X, f, x01)  # xcs: (1,N_BINS,1,H,W), preds: (1,N_BINS)
    if out_png is None:
        out_png = SINGLE_DIR / f"{Path(path).stem}_strip.png"
    save_strip_png(str(out_png), x01[0], xcs[0], preds[0], fx[0].item())

    centers = [(b + 0.5) / N_BINS for b in range(N_BINS)]
    pred_list = preds[0].tolist()
    out = dict(path=path, f_x=fx[0].item(), centers=centers, f_xc=pred_list, strip=str(out_png))

    if show:
        print(f"Ảnh: {path}")
        known = sc.loc[sc["path"] == path, "label_name"]
        if len(known):
            print(f"Nhãn thật: {known.iloc[0]}")
        print(f"f(x) = {out['f_x']:.4f}  (bin hiện tại = {int(score_to_bin_np(np.array([out['f_x']]))[0])})")
        display(IPImage(filename=str(out_png), width=1800))

        fig, ax = plt.subplots(figsize=(4.5, 4))
        ax.plot(centers, pred_list, "o-", label="f(x_c)")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="lý tưởng f(x_c) = c")
        ax.axhline(out["f_x"], color="tab:red", ls=":", label="f(x) gốc")
        ax.set_xlabel("điều kiện c (tâm bin)"); ax.set_ylabel("f(x_c)")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
    return out

### 9.2 Chọn ảnh đầu vào

- Đặt `SINGLE_IMAGE_PATH` là đường dẫn một ảnh bất kỳ (ví dụ ảnh trong Kaggle Dataset hoặc ảnh tải lên ở `/kaggle/working`). Ảnh sẽ được crop giữa và resize về 256x256 bằng `load_image` như khi train.
- Nếu để `None`, notebook chọn ảnh từ tập test theo `SINGLE_GROUP` (`neg` | `mild` | `pos` | `any`) và `SINGLE_INDEX`.

In [ ]:
# đường dẫn ảnh
SINGLE_IMAGE_PATH = None 
SINGLE_GROUP, SINGLE_INDEX = "pos", 0

GROUPS = {   # giống generate.py
    "neg": lambda s: s < 0.2,
    "mild": lambda s: (s >= 0.4) & (s <= 0.6),
    "pos": lambda s: s > 0.8,
    "any": lambda s: s >= 0.0,
}

if SINGLE_IMAGE_PATH is None:
    pool = sc[GROUPS[SINGLE_GROUP](sc["score"])].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    assert len(pool), f"Không có ảnh nào thuộc nhóm '{SINGLE_GROUP}'"
    SINGLE_IMAGE_PATH = pool.loc[SINGLE_INDEX % len(pool), "path"]
assert Path(SINGLE_IMAGE_PATH).exists(), f"Không tìm thấy ảnh: {SINGLE_IMAGE_PATH}"
print("Ảnh đầu vào:", SINGLE_IMAGE_PATH)

### 9.3 Sinh dải ảnh phản thực cho ảnh đó

In [ ]:
r = explain_image(SINGLE_IMAGE_PATH)
display(pd.DataFrame({"c (tâm bin)": r["centers"], "f(x_c)": r["f_xc"]}).round(4).T)

### 9.4 (Tùy chọn) Chạy cho nhiều ảnh riêng lẻ

In [ ]:
# danh sách ảnh
IMAGE_LIST = [
    # "/kaggle/input/my-images/ca_01.png",
    # "/kaggle/input/my-images/ca_02.png",
]
summary = []
for p in IMAGE_LIST:
    r = explain_image(p)
    summary.append({"path": p, "f_x": r["f_x"], **{f"f_xc_bin{b}": v for b, v in enumerate(r["f_xc"])}})
if summary:
    pd.DataFrame(summary).to_csv(SINGLE_DIR / "single_predictions.csv", index=False)
    display(pd.DataFrame(summary).round(3))

## 10. Lưu thông tin lần kiểm thử

In [ ]:
(OUT_DIR / "test_info.json").write_text(json.dumps(dict(
    explainer=explainer_src, classifier=classifier_src, fid_net=fid_src,
    target=TARGET, test_splits=TEST_SPLITS, tau=TAU, max_images=MAX_IMAGES, seed=SEED,
    code_repo=GITHUB_REPO_URL, code_commit=GIT_COMMIT), indent=2, ensure_ascii=False))
print("Đã lưu:", OUT_DIR / "test_info.json")
print("Kết quả nằm trong:", OUT_DIR)